**Lab 07 MODEL EVALUATION AND CROSS-VALIDATION**

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
# Load dataset
input_path = Path("data/processed/olist_orders_feature_engineered.csv")
if input_path.exists():
  df = pd.read_csv(input_path)
  print("Loaded real dataset. Shape:", df.shape)
else:
  print("Real file not found at path; generating realistic Olist mock data...")
  np.random.seed(42)
  n_samples = 5000
  df = pd.DataFrame({
      "freight_value": np.random.exponential(scale=20, size=n_samples),
      "price": np.random.exponential(scale=120, size=n_samples),
      "product_weight_g": np.random.normal(loc=2000, scale=800, size=n_samples)
      .clip(100),
      "estimated_days": np.random.randint(5, 30, size=n_samples),
      "distance_km": np.random.exponential(scale=600, size=n_samples),
      "seller_prep_time": np.random.randint(1, 7, size=n_samples),
  })
  # Generate imbalanced target (~8% late orders)
  late_prob = (
      1
      / (
          1
          + np.exp(
              -(
                  0.002 * df["distance_km"]
                  + 0.15 * df["seller_prep_time"]
                  - 0.1 * df["estimated_days"]
                  - 2.5
              )
          )
      )
  )
  df["is_late_delivery"] = (np.random.rand(n_samples) < late_prob).astype(int)
target = "is_late_delivery"
print("Shape:", df.shape)
print("\nTarget Class Distribution:")
print(df[target].value_counts())
print("\nTarget Class Proportions:")
print(df[target].value_counts(normalize=True))

Real file not found at path; generating realistic Olist mock data...
Shape: (5000, 7)

Target Class Distribution:
is_late_delivery
0    4320
1     680
Name: count, dtype: int64

Target Class Proportions:
is_late_delivery
0    0.864
1    0.136
Name: proportion, dtype: float64


In [2]:
y = df[target]
X = df.drop(columns=[target])
X = X.select_dtypes(include=np.number)
X = X.fillna(X.median())
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5000, 6)
y shape: (5000,)


In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(
    f"Train set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples"
)

Train set: 4000 samples, Test set: 1000 samples


In [4]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [5]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"True Negatives (TN):  {cm[0, 0]}")
print(f"False Positives (FP): {cm[0, 1]}")
print(f"False Negatives (FN): {cm[1, 0]}")
print(f"True Positives (TP):  {cm[1, 1]}")

Confusion Matrix:
[[691 173]
 [ 39  97]]
True Negatives (TN):  691
False Positives (FP): 173
False Negatives (FN): 39
True Positives (TP):  97


In [6]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.7880
Precision: 0.3593
Recall:    0.7132
F1-score:  0.4778


In [7]:
from sklearn.metrics import classification_report
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, zero_division=0))
# --- CELL 8: Part K - ROC-AUC Score ---
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")


--- Classification Report ---
              precision    recall  f1-score   support

           0       0.95      0.80      0.87       864
           1       0.36      0.71      0.48       136

    accuracy                           0.79      1000
   macro avg       0.65      0.76      0.67      1000
weighted avg       0.87      0.79      0.81      1000

ROC-AUC Score: 0.8355


In [8]:
print("\n--- Threshold Analysis ---")
for t in [0.30, 0.40, 0.50, 0.70]:
  preds_t = (y_prob >= t).astype(int)
  p = precision_score(y_test, preds_t, zero_division=0)
  r = recall_score(y_test, preds_t, zero_division=0)
  f = f1_score(y_test, preds_t, zero_division=0)
  print(
      f"Threshold {t:.2f} -> Precision: {p:.3f} | Recall: {r:.3f} | F1: {f:.3f}"
  )


--- Threshold Analysis ---
Threshold 0.30 -> Precision: 0.224 | Recall: 0.890 | F1: 0.358
Threshold 0.40 -> Precision: 0.286 | Recall: 0.794 | F1: 0.421
Threshold 0.50 -> Precision: 0.359 | Recall: 0.713 | F1: 0.478
Threshold 0.70 -> Precision: 0.533 | Recall: 0.537 | F1: 0.535


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1_scores = cross_val_score(
    model, X_train, y_train, cv=cv, scoring="f1"
)
print("\n5-Fold F1 Scores:", np.round(cv_f1_scores, 4))
print(f"CV Mean F1: {cv_f1_scores.mean():.4f}")
print(f"CV Std F1:  {cv_f1_scores.std():.4f}")


5-Fold F1 Scores: [0.48   0.4451 0.4286 0.4334 0.4969]
CV Mean F1: 0.4568
CV Std F1:  0.0269


In [10]:
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_results = cross_validate(
    model, X_train, y_train, cv=cv, scoring=scoring
)
print("\n--- CV Multi-Metric Summary ---")
for metric in scoring:
  vals = cv_results["test_" + metric]
  print(f"{metric:12s} Mean: {vals.mean():.3f} | Std: {vals.std():.3f}")


--- CV Multi-Metric Summary ---
accuracy     Mean: 0.777 | Std: 0.014
precision    Mean: 0.342 | Std: 0.022
recall       Mean: 0.689 | Std: 0.035
f1           Mean: 0.457 | Std: 0.027
roc_auc      Mean: 0.825 | Std: 0.018


In [11]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(max_iter=1000, class_weight="balanced"),
        ),
    ]
)

In [12]:
cv_pipe_scores = cross_val_score(
    pipeline, X_train, y_train, cv=cv, scoring="f1"
)
print("\n--- Pipeline Cross-Validation ---")
print(f"Pipeline CV Mean F1: {cv_pipe_scores.mean():.4f}")
print(f"Pipeline CV Std F1:  {cv_pipe_scores.std():.4f}")


--- Pipeline Cross-Validation ---
Pipeline CV Mean F1: 0.4574
Pipeline CV Std F1:  0.0273


In [14]:
pipeline.fit(X_train, y_train)
final_pred = pipeline.predict(X_test)
print("\n--- Final Test Set Evaluation (Classification Report) ---")
print(classification_report(y_test, final_pred, zero_division=0))


--- Final Test Set Evaluation (Classification Report) ---
              precision    recall  f1-score   support

           0       0.95      0.80      0.87       864
           1       0.36      0.71      0.48       136

    accuracy                           0.79      1000
   macro avg       0.65      0.76      0.67      1000
weighted avg       0.87      0.79      0.81      1000

